# RSNA Knee — ResNet18 slice embeddings + LightGBM
GPU推奨。Internet ONでImageNet重みを取得する。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
SRC = Path('/kaggle/working/repo/rsna-knee-abnormality-detection/src')
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb','/kaggle/working/repo'], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS, add_pseudo_labels
import cnn_features as cf, cv_eval, torch
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
train = pd.read_csv(COMP/'train.csv'); tseries = pd.read_csv(COMP/'train_series.csv')
test = pd.read_csv(COMP/'test.csv');   teseries = pd.read_csv(COMP/'test_series.csv')
print(train.shape, tseries.shape, test.shape)
print(train[LABELS].notna().sum())   # 正解ラベルがあるstudy数
train['has_true'] = train['ACL'].notna()   # 正解ラベルあり(疑似ラベルではない)
train = add_pseudo_labels(train)
print(train[LABELS].mean().round(3))

In [ ]:
N = 600  # 疑似ラベルstudyの標本数(正解ラベル付きは常に全件)。まず小さく動作確認
rest = train[~train.has_true]
tr = pd.concat([train[train.has_true], rest.sample(min(N, len(rest)), random_state=0)]).reset_index(drop=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
model = cf.load_model(dev)
Xtr = cf.extract(tr, tseries, COMP/'train_series', model, dev); print(Xtr.shape)
np.save('Xtr.npy', Xtr)

In [ ]:
oof = cv_eval.cv_report(Xtr, tr, LABELS)

In [ ]:
Xte = cf.extract(test, teseries, COMP/'test_series', model, dev)
sub = cv_eval.fit_predict(Xtr, tr, Xte, LABELS, test['StudyInstanceUID'])
sub.to_csv('submission.csv', index=False); sub.head()